In [1]:
!pip install langchain
!pip install pandas
!pip install sklearn

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [2]:
!pip install openai

In [3]:
! pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.3/409.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: SQLAlchemy
    Found existing installation: SQLAlchemy 2.0.36
    Uninstalling SQLAlchemy-2.0.36:
      Successfully uninstalled SQLAlchemy-2.0.36
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.15
    Uninstalling langchain-core-0.3.15:
      Successfully uninstalled langchain-core-0.3.15


In [1]:
import pandas as pd
from sklearn.metrics import mean_squared_error
from langchain import LLMChain

# 사주명식과 MBTI 데이터 수집 및 전처리

data = {
    'birth_date': ['1990-01-01', '1995-05-15', '2000-12-30'],
    'mbti_type': ['ENTJ', 'INFP', 'ESFP']
}
df = pd.DataFrame(data)
df

,birth_date,mbti_type
0,1990-01-01,ENTJ
1,1995-05-15,INFP
2,2000-12-30,ESFP


In [12]:
# data['birth_date']
# df['birth_date'].values
# df['birth_date'].values[0]
# df['birth_date'].values[0].split('-')[0]
int(df['birth_date'].values[0].split('-')[0])

1990

In [16]:
# 사주명식 계산 함수 (간단한 예제)
def calculate_saju(birth_date):
    # 실제 사주명식 계산 로직은 복잡할 수 있음
    # 여기서는 단순히 생년월일의 연도만 사용
    year = int(birth_date.split('-')[0])
    if year % 2 == 0:
        return '양'
    else:
        return '음'

# 데이터프레임에 사주명식 열 추가
df['saju'] = df['birth_date'].apply(calculate_saju)
df

,birth_date,mbti_type,saju
0,1990-01-01,ENTJ,양
1,1995-05-15,INFP,음
2,2000-12-30,ESFP,양


In [13]:
# 사주명식과 MBTI 매핑
saju_to_mbti = {
    '양': '외향형',
    '음': '내향형'
}

In [14]:
import os

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('temp')

In [17]:
# 랭체인 기반 챗봇 개발
from langchain import OpenAI, LLMChain
from langchain.prompts import PromptTemplate

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["birth_date"],
    template="생년월일 {birth_date}에 따른 사주명식과 MBTI 성격 유형을 설명해줘."
)

# Initialize the OpenAI model
# llm = OpenAI(model_name="text-davinci-003", api_key="your_openai_api_key")
llm = OpenAI(model_name="text-davinci-003")

# Define the chain with the prompt and model
class SajuMBTIChain(LLMChain):
    # 클래스 생성자(instance 초기)
    def __init__(self, prompt, llm):
        super().__init__(prompt=prompt, llm=llm)
        # 부모 클래스(LLMChain) 의 생성자 호출>>상속된 속성 초기

    def generate_response(self, birth_date):
        # birth_date(생년월일)을 입력으로 받음
        saju = calculate_saju(birth_date)
        mbti = saju_to_mbti[saju]
        return f"당신의 사주명식은 {saju}이며, 이는 MBTI 성격 유형의 {mbti}와 관련이 있습니다."

# Example usage
chatbot = SajuMBTIChain(prompt=prompt_template, llm=llm)
birth_date = '1978-10-26'
response = chatbot.generate_response(birth_date)
print(response)




당신의 사주명식은 양이며, 이는 MBTI 성격 유형의 외향형와 관련이 있습니다.


In [18]:
from transformers import pipeline
from langchain import LLMChain
from langchain_core.runnables.base import Runnable
# 커스텀 실행 가능 객체 정의
from langchain.prompts import PromptTemplate

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["birth_date"],
    template="생년월일 {birth_date}에 따른 사주명식과 MBTI 성격 유형을 설명해줘."
)

# Define the custom Runnable wrapper for Hugging Face pipeline
class HuggingFaceRunnable(Runnable):
    def __init__(self, hf_pipeline):
        self.hf_pipeline = hf_pipeline

    def invoke(self, prompt):
        result = self.hf_pipeline(prompt)[0]['generated_text']
        return result
        # 프롬프트를 받아 허깅페이스 모델 실행
        # >> 텍스트 반환

# Initialize the Hugging Face model
hf_pipeline = pipeline("text-generation", model="gpt2")

# Wrap the pipeline with HuggingFaceRunnable
llm = HuggingFaceRunnable(hf_pipeline)

# Define the chain with the prompt and model
class SajuMBTIChain_hf(LLMChain):
    def __init__(self, prompt, llm):
        super().__init__(prompt=prompt, llm=llm)

    def generate_response(self, birth_date):
        saju = calculate_saju(birth_date)
        mbti = saju_to_mbti[saju]
        return f"당신의 사주명식은 {saju}이며, 이는 MBTI 성격 유형의 {mbti}와 관련이 있습니다."

# Example usage
chatbot = SajuMBTIChain_hf(prompt=prompt_template, llm=llm)
birth_date = '1990-01-01'
response = chatbot.generate_response(birth_date)
print(response)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


당신의 사주명식은 양이며, 이는 MBTI 성격 유형의 외향형와 관련이 있습니다.


In [19]:
!pip install gradio

In [20]:
import gradio as gr

# Gradio 인터페이스 설정
def chatbot_interface(birth_date):
    return chatbot.generate_response(birth_date)

iface = gr.Interface(
    fn=chatbot_interface,
    inputs=gr.Textbox(label='생년월일 (예: 1990-01-01)'),
    outputs="text",
    examples=['1990-01-01', '1995-05-15', '2000-12-30']
)

iface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3bbdc5029c596681de.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
